# Chapter 5: Does it meet Okanogan's two requirements?

1. Deliver an item to a target location.
2. Never hit a shelf.

**Evaluation:** the best configuration (replay + target network, seed 0), 300 unseen episodes, compared with a random driver and a shortest-path (BFS) reference that has full knowledge of the map.

In [1]:
import sys, json; sys.path.insert(0, "..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
from src.warehouse_env import WarehouseEnv, evaluate_policy, MOVES
abl = json.load(open("../results/ablation.json"))
runs = pd.DataFrame([dict(variant=r["variant"], seed=r["seed"], **r["eval"]) for r in abl["runs"]])
import torch
from src.dqn import QNet, greedy_action
env = WarehouseEnv()
q = QNet(11, 4); q.load_state_dict(torch.load("../models/dqn_replay_target_seed0.pt")); q.eval()

def expert(o):                                  # shortest-path reference (knows the map)
    goal = env._goal(); best = None
    for a, (dx, dy) in enumerate(MOVES):
        nxt = (env.robot[0] + dx, env.robot[1] + dy)
        if env.blocked(*nxt): continue
        d = env._dist(nxt, goal)
        if best is None or d < best[0]: best = (d, a)
    return best[1]

res = {"random": evaluate_policy(env, lambda o: env.action_space.sample()),
       "trained DQN": evaluate_policy(env, lambda o: greedy_action(q, o)),
       "shortest-path reference": evaluate_policy(env, expert)}
table = pd.DataFrame(res).T.round(3); print(table.to_string())
json.dump(res, open("../results/final_eval.json", "w"))

                         delivery_rate  collision_rate  bumps_per_episode  path_efficiency
random                           0.053           1.000             27.140            0.268
trained DQN                      0.360           0.037              2.697            0.985
shortest-path reference          1.000           0.000              0.000            1.000


### Per-seed view (all seeds of the winning variant)

In [2]:
print(runs[runs.variant == "replay+target"].round(3).to_string(index=False))

      variant  seed  delivery_rate  collision_rate  bumps_per_episode  path_efficiency
replay+target     0          0.360           0.037              2.697            0.985
replay+target     1          0.480           0.000              0.000            0.982
replay+target     2          0.573           0.007              0.480            0.989


## One trained episode

In [3]:
import imageio.v2 as imageio
from matplotlib.patches import Circle, Rectangle
def frame(env, t):
    fig, ax = plt.subplots(figsize=(3.6, 3.6)); ax.imshow(env.grid, cmap="Greys", vmin=0, vmax=1.6); ax.axis("off")
    if not env.has_item: ax.add_patch(Rectangle((env.item[0]-.3, env.item[1]-.3), .6, .6, color="#f2c811"))
    ax.add_patch(Rectangle((env.target[0]-.35, env.target[1]-.35), .7, .7, fill=False, ec="#2e9e5b", lw=2.5))
    ax.add_patch(Circle(env.robot, .32, color="#1f77b4"))
    if env.has_item: ax.add_patch(Rectangle((env.robot[0]-.12, env.robot[1]-.12), .24, .24, color="#f2c811"))
    ax.set_title(f"step {t}  ·  {'carrying item' if env.has_item else 'heading to item'}", fontsize=9)
    fig.canvas.draw(); img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy(); plt.close(fig); return img
for seed in range(60_000, 60_100):
    o, _ = env.reset(seed=seed); frames = [frame(env, 0)]; done = False; t = 0
    while not done:
        o, r, term, trunc, info = env.step(greedy_action(q, o)); t += 1; frames.append(frame(env, t)); done = term or trunc
    if info["delivered"] and env.collisions == 0 and t >= 12: break
imageio.mimsave("../docs/img/ch5_trained_robot.gif", frames + [frames[-1]] * 4, duration=0.25, loop=0)
print(f"seed {seed}: delivered in {t} steps, collisions {env.collisions}")

seed 60000: delivered in 18 steps, collisions 0


## Why does it fail? Loops, and what noise and a safety mask change
During training the agents looked much better (80-96% delivery) than in this greedy evaluation. Training used 5% random actions. Three checks: (a) are greedy failures loops, (b) what does random noise do to delivery *and* collisions, (c) what if blocked moves are masked out, i.e. a simple safety shield that uses the 'blocked neighbour' flags the robot already senses?

In [4]:
import random
random.seed(0)
# (a) greedy failures: does the robot revisit one (position, holding) state >= 10 times?
fail = stuck = 0
for ep in range(300):
    o, _ = env.reset(seed=50_000 + ep); seen = {}; done = False
    while not done:
        o, r, term, trunc, info = env.step(greedy_action(q, o)); done = term or trunc
        k = (env.robot, env.has_item); seen[k] = seen.get(k, 0) + 1
    if not info["delivered"]:
        fail += 1; stuck += max(seen.values()) >= 10
print(f"(a) greedy failures: {fail}/300 episodes, of which loops (same state >=10 times): {stuck}")

# (b) epsilon noise
rows = []
for eps in [0.0, 0.02, 0.05, 0.1]:
    pol = lambda o, eps=eps: env.action_space.sample() if random.random() < eps else greedy_action(q, o)
    rows.append(dict(policy=f"greedy + {int(eps*100)}% random actions", **evaluate_policy(env, pol)))

# (c) safety mask: never choose an action whose target cell is blocked (flags are o[7:11], same order as actions)
def masked(o, eps=0.0):
    ok = [a for a in range(4) if o[7 + a] == 0]
    if random.random() < eps: return random.choice(ok)
    with torch.no_grad(): qv = q(torch.as_tensor(o)).numpy()
    return max(ok, key=lambda a: qv[a])
rows.append(dict(policy="greedy + safety mask", **evaluate_policy(env, lambda o: masked(o))))
rows.append(dict(policy="greedy + safety mask + 5% random (safe) actions", **evaluate_policy(env, lambda o: masked(o, 0.05))))
rows.append(dict(policy="greedy + safety mask + 10% random (safe) actions", **evaluate_policy(env, lambda o: masked(o, 0.10))))
diag = pd.DataFrame(rows).set_index("policy").round(3); print("\n(b),(c)\n" + diag.to_string())
diag.to_csv("../results/ch5_diagnosis.csv")

(a) greedy failures: 192/300 episodes, of which loops (same state >=10 times): 192



(b),(c)
                                                  delivery_rate  collision_rate  bumps_per_episode  path_efficiency
policy                                                                                                             
greedy + 0% random actions                                0.360           0.037              2.697            0.985
greedy + 2% random actions                                0.417           0.290              2.450            0.882
greedy + 5% random actions                                0.507           0.470              3.113            0.778
greedy + 10% random actions                               0.573           0.673              3.053            0.677
greedy + safety mask                                      0.373           0.000              0.000            0.984
greedy + safety mask + 5% random (safe) actions           0.560           0.000              0.000            0.752
greedy + safety mask + 10% random (safe) actions          0.623

## Takeaways
**Verdict against Okanogan's requirements (300 unseen episodes):**
- **Requirement 1, deliver the item: not met.** The best-seed DQN delivers **36%** greedily (seeds of the same variant: 36%, 48%, 57%), versus ~5% for random and 100% for the shortest-path reference. When it does deliver, the route is near-optimal (path efficiency 0.985).
- **Why it fails:** all 192 failed greedy episodes are loops (the robot revisits one state 10+ times). Random noise breaks loops (delivery up to 57% at 10% random actions) but makes the robot bump shelves in 29-67% of episodes.
- **Requirement 2, never hit a shelf: not met by learning alone.** Greedy bumped in 3.7% of episodes (versus 100% for random). A simple **safety mask** that forbids moves into blocked cells makes bumps zero by construction, and with 10% safe random actions delivery reaches 62%, still short of 100%.
- Training-time delivery (80-96%) was misleading: it was measured with exploration noise and ignores bumps. Evaluate greedily and count safety events separately.

**Next steps (not yet tried):** give the agent memory of recent positions to break loops, Double DQN or longer training with a decaying learning rate, and more seeds.